# Lab 01 — Foundations & Classical Agents

Kiel University · Agentic AI (infAgAI-01a) · Winter 2026

**Learning objectives.** After this lab you can:

- set up the Python toolchain and verify that a local open-source model serves chat completions via **Ollama**,
- issue bare chat completions and steer them with **system prompts** and **temperature**,
- implement a **simple reflex agent** (condition–action rules) for a toy source-triage task,
- observe precisely **where hand-written rules break**, and name the responsible environment properties in lecture vocabulary,
- replace the rule table with a **single LLM call** (a learned policy) and compare behaviour,
- describe the course's **research agent** in PEAS and BDI vocabulary.

*Estimated effort: 90–120 minutes. Fill in the gaps from Part B onwards; fold-out solutions sit below most cells. Tasks marked 📝 belong in your lab report and come without solutions.*


## Theory recap — what makes software an agent?

### The classical definition (Russell & Norvig)

An **agent** is anything that can be viewed as *perceiving* its environment through **sensors** and *acting* upon that environment through **actuators**. A thermostat qualifies; so does a Mars rover. The **agent function** is the abstract mapping $f: P^* \to A$ from complete percept sequences to actions — the specification of behaviour. The **agent program** is its concrete, finite implementation on an architecture. A **rational agent** selects, for each percept sequence, the action *expected* to maximise its **performance measure**, given the evidence so far — rationality is not omniscience. To specify a task, use **PEAS**: Performance measure, Environment, Actuators, Sensors. **Autonomy**, in the classical sense, means behaviour driven by the agent's own percepts rather than only by assumptions the designer baked in.

### Task environments shape agent design

Before designing an agent, classify its environment along six dimensions: fully vs. **partially observable**, **deterministic** vs. stochastic, **episodic** vs. sequential, **static** vs. dynamic, **discrete** vs. continuous, **single-** vs. **multi-agent**. The open web — the environment of our research agent — is partially observable, stochastic, sequential, dynamic, discrete in actions but with unbounded text percepts, and multi-agent (search ranking, paywalls, adversarial SEO). That is exactly the corner of the taxonomy that defeated classical planners.

### Five agent programs — and today's star, the simple reflex agent

Russell & Norvig's hierarchy of increasing internal structure: **simple reflex** (condition–action rules on the current percept only), **model-based reflex** (adds internal state tracking the unobserved parts of the world), **goal-based** (predicts future states, plans), **utility-based** (ranks outcomes by a utility function), and **learning** agents. Today we build a simple reflex agent — fast, trivially implementable, and doomed to fail wherever history matters or the world leaves the designer's assumptions.

### The modern LLM agent

The course's working definition: *an agent is a system in which an LLM decides, in a loop, which actions to take next in pursuit of a goal — calling tools and observing their results.* Four components: **model** (the LLM as policy), **tools**, **loop**, **goal** — and the **state** is the growing context window. The LLM itself is stateless between calls; the conversation is the agent's memory.

### Autonomy is a dial — and agent-washing is real

Between chatbot and agent lies a spectrum: **chatbot → augmented LLM → workflow → agent**. Each step to the right hands more control-flow decisions from developer to model; autonomy is a dial the designer sets, not a virtue. The five-question **agent-washing checklist**: (1) who picks the next step at runtime — model or script? (2) does it execute tools and observe results, or only generate text? (3) can the step count vary with the task, under stop conditions? (4) does it replan after a failed action, or abort? (5) do the autonomy claims match the deployed reality? Today's lab sits deliberately at the **far left of the dial**: bare completions and a rule-based reflex agent — the baseline from which every later capability of this course is an explicit, visible addition.

### Classical architectures in one paragraph

**Deliberative** agents (Shakey / STRIPS, 1971) sense, plan over an explicit symbolic world model, then act — powerful but slow and brittle. **Reactive** agents (Brooks' subsumption architecture, 1986) stack fast stimulus–response layers with no world model at all: *"the world is its own best model."* **Hybrid / BDI** agents (PRS, 1987) keep **beliefs** (current, possibly wrong information), **desires** (goal states, possibly conflicting) and **intentions** (desires the agent has *committed* to; they persist and constrain replanning), with plans drawn from a hand-written library. All three run the **perceive–reason–act loop**, and the loop *closes through the world*: the agent observes what the environment actually did, not what it intended. The recurring tension — deliberation is powerful but slow, reaction is fast but short-sighted — is as alive in 2026 as it was in 1986.


> **Q:** Distinguish the *agent function* from the *agent program*.
<details><summary>Click for answer</summary>

The agent function is an abstract mathematical mapping from complete percept histories to actions — a specification of behaviour in every conceivable situation. The agent program is the concrete, finite implementation of (an approximation of) that function, running on a particular architecture. The same distinction applies to LLM agents: the intended behaviour versus the prompt, model and loop code that realise it.

</details>


## Part A — Setup & Ollama connectivity check

You need Python ≥ 3.10 with `ollama`, `numpy`, `pandas`, `matplotlib` (and optionally `ipywidgets`):

```bash
pip install ollama numpy pandas matplotlib ipywidgets
```

You also need the [Ollama](https://ollama.com) runtime serving a local open-source model:

```bash
ollama serve                 # if it is not already running as a service
ollama pull qwen2.5:7b       # any tool-capable 7–30B model works
```

The cell below imports everything and performs a friendly connectivity check. Everything in this notebook except the LLM cells also works without Ollama.


In [ ]:
import os

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

MODEL = os.environ.get("OLLAMA_MODEL", "qwen2.5:7b")   # any tool-capable 7-30B model works

try:
    import ollama
    resp = ollama.chat(model=MODEL,
                       messages=[{"role": "user", "content": "Reply with exactly one word: ready"}])
    print(f"Ollama is reachable. Model '{MODEL}' answered:", resp["message"]["content"].strip())
except Exception as e:
    print("Could not talk to Ollama:", e)
    print("Start Ollama with `ollama serve` and pull the model with `ollama pull qwen2.5:7b`,")
    print("then re-run this cell. Everything except the LLM cells works without Ollama.")


## Part B — Your first bare model call

We start at the **far-left notch of the autonomy dial**: a chatbot — one model call, text in, text out, no tools, no loop. The `ollama.chat` API takes a list of `messages`, each with a `role`:

- `system` — persistent instructions that frame *how* the model should behave (its persona, format, constraints),
- `user` — the actual request,
- `assistant` — earlier model replies (used later, when we manage conversation state ourselves).

The `options` dict controls sampling. **Temperature** scales the randomness of token sampling: at $T=0$ the model (nearly) always picks the most probable token — good for classifiers and reproducibility; higher values flatten the distribution — more diverse, less reliable output.

Fill in the gaps: the two role names, the messages argument, and a low-ish temperature (e.g. $0.2$).


In [ ]:
SYSTEM_PROMPT = "You are a concise research assistant. Answer in at most two sentences."

messages = [
    {"role": ___, "content": SYSTEM_PROMPT},
    {"role": ___, "content": "In one sentence: what is a simple reflex agent?"},
]

try:
    resp = ollama.chat(model=MODEL, messages=___, options={"temperature": ___})
    print(resp["message"]["content"])
except Exception as e:
    print("Ollama call failed:", e)
    print("Start Ollama with `ollama serve` and pull the model with `ollama pull qwen2.5:7b`.")


<details>
<summary><b>Click here for the solution</b></summary>

```python
SYSTEM_PROMPT = "You are a concise research assistant. Answer in at most two sentences."

messages = [
    {"role": "system", "content": SYSTEM_PROMPT},
    {"role": "user", "content": "In one sentence: what is a simple reflex agent?"},
]

try:
    resp = ollama.chat(model=MODEL, messages=messages, options={"temperature": 0.2})
    print(resp["message"]["content"])
except Exception as e:
    print("Ollama call failed:", e)
    print("Start Ollama with `ollama serve` and pull the model with `ollama pull qwen2.5:7b`.")
```

</details>


> **Q:** How does the chatbot end of the autonomy spectrum connect to this lab?
<details><summary>Click for answer</summary>

Lab 1 deliberately starts at the leftmost position: you set up a local open-source model and issue bare completions — a single call, no tools, no loop. The pedagogical purpose is to experience the baseline from which every later capability is an explicit, visible addition: tools in Unit 3, loops and patterns in Unit 2, orchestration in Unit 5. Feeling each notch of the dial prevents treating agent frameworks as black magic.

</details>


> **Q:** Why does this course run its labs on local open-source models rather than commercial APIs? *(not exam-relevant)*
<details><summary>Click for answer</summary>

Local models expose everything an API hides — the exact prompt, the raw token stream, latencies and failure modes — and allow unlimited experimentation at zero marginal cost, which suits a course about mechanisms. The trade-off is capability: smaller open models fail more often and more visibly than frontier models. For learning purposes this is partly a feature, since the course is centrally concerned with understanding and engineering around failure modes.

</details>


## Part C — A hand-coded simple reflex agent: source triage

Our toy task is a miniature of the research agent's **EVALUATE** step (brief → search → **evaluate sources** → draft → revise). The brief: *"the current state of solid-state battery manufacturing."* A search has returned 20 result records, stored in `data/search_results.csv`:

| column | meaning |
|---|---|
| `domain`, `title`, `snippet` | the **percept** — one search result, as the agent sees it |
| `gold` | the correct **action**: `KEEP` (useful, trustworthy, relevant) or `DISCARD` |
| `subset` | `easy` (the situations the rule designer anticipated) or `tricky` (see Part D) |

The agent is a **simple reflex agent**: it maps the *current percept only* to an action via ordered condition–action rules — no memory, no goals, no model of the world. First, load the data.


In [ ]:
DATA = "data/search_results.csv"
if not os.path.exists(DATA):                       # the solution notebook lives in _solutions/
    DATA = "../Lab01_Foundations/data/search_results.csv"

df = pd.read_csv(___)
easy   = df[df["subset"] == ___].reset_index(drop=True)
tricky = df[df["subset"] == ___].reset_index(drop=True)
print(f"{len(df)} search results: {len(easy)} easy, {len(tricky)} tricky")
df.head()


<details>
<summary><b>Click here for the solution</b></summary>

```python
DATA = "data/search_results.csv"
if not os.path.exists(DATA):                       # the solution notebook lives in _solutions/
    DATA = "../Lab01_Foundations/data/search_results.csv"

df = pd.read_csv(DATA)
easy   = df[df["subset"] == "easy"].reset_index(drop=True)
tricky = df[df["subset"] == "tricky"].reset_index(drop=True)
print(f"{len(df)} search results: {len(easy)} easy, {len(tricky)} tricky")
df.head()
```

</details>


### The rule table

The rules below were written by inspecting the *easy* subset — exactly how reflex agents are built in practice: the designer looks at anticipated situations and writes conditions for them. Note the ordering: a trusted origin wins over everything, spam vocabulary discards, quality vocabulary keeps, and the default is conservative. Fill in the gaps.


In [ ]:
TRUSTED_SITES   = ("nature.com", "sciencedirect.com", "ieee.org", "arxiv.org")
SPAM_MARKERS    = ("buy now", "% off", "discount code", "subscribe", "giveaway")
QUALITY_MARKERS = ("peer-reviewed", "journal", "study", "dataset")

def reflex_agent(percept):
    '''Simple reflex agent: condition-action rules on the CURRENT percept only.'''
    domain = percept["domain"].lower()
    text = (percept["title"] + " " + percept["snippet"]).lower()

    if domain.endswith((".edu", ".gov")) or domain in ___:      # rule 1: trusted origin
        return ___
    if any(marker in text for marker in ___):                   # rule 2: commercial spam
        return "DISCARD"
    if any(marker in text for marker in QUALITY_MARKERS):           # rule 3: quality vocabulary
        return ___
    return ___                                                  # default: be conservative

# quick smoke test on the first easy percept
print(reflex_agent(easy.iloc[0].to_dict()), "| expected:", easy.iloc[0]["gold"])


<details>
<summary><b>Click here for the solution</b></summary>

```python
TRUSTED_SITES   = ("nature.com", "sciencedirect.com", "ieee.org", "arxiv.org")
SPAM_MARKERS    = ("buy now", "% off", "discount code", "subscribe", "giveaway")
QUALITY_MARKERS = ("peer-reviewed", "journal", "study", "dataset")

def reflex_agent(percept):
    '''Simple reflex agent: condition-action rules on the CURRENT percept only.'''
    domain = percept["domain"].lower()
    text = (percept["title"] + " " + percept["snippet"]).lower()

    if domain.endswith((".edu", ".gov")) or domain in TRUSTED_SITES:      # rule 1: trusted origin
        return "KEEP"
    if any(marker in text for marker in SPAM_MARKERS):                   # rule 2: commercial spam
        return "DISCARD"
    if any(marker in text for marker in QUALITY_MARKERS):           # rule 3: quality vocabulary
        return "KEEP"
    return "DISCARD"                                                  # default: be conservative

# quick smoke test on the first easy percept
print(reflex_agent(easy.iloc[0].to_dict()), "| expected:", easy.iloc[0]["gold"])
```

</details>


<details>
<summary><b>Click here for a detailed code explanation</b></summary>

The function implements exactly the simple-reflex schema from the lecture — `percept -> action` via ordered condition–action rules:

- **Rule order is part of the design.** A trusted origin (rule 1) is checked first, so a `.gov` page is kept even if its snippet happened to contain a spam word. Swapping rules 2 and 3 would change behaviour on percepts that contain both spam and quality vocabulary.
- **The percept is flattened to lowercase text** so the string-matching conditions are case-insensitive. Everything the agent "knows" about language is these three tuples of markers.
- **There is no state anywhere.** Two calls with the same percept always return the same action, and no call can remember an earlier one — the defining property (and the defining weakness) of a simple reflex agent.
- The `to_dict()` in the smoke test converts one pandas row into a plain dict — our percept format.

</details>


### Evaluate on the anticipated situations

We treat this as an *episodic* evaluation for now — each percept judged independently against `gold`. Complete the evaluation helper; you will reuse it for every agent in this lab.


In [ ]:
def evaluate(agent_fn, frame, name):
    correct = 0
    for _, row in frame.iterrows():
        action = agent_fn(___)              # the percept is one row, as a dict
        correct += int(action == row[___])
    accuracy = correct / ___
    print(f"{name}: {correct}/{len(frame)} correct (accuracy {accuracy:.2f})")
    return accuracy

acc_reflex_easy = evaluate(reflex_agent, easy, "reflex agent, easy subset")


<details>
<summary><b>Click here for the solution</b></summary>

```python
def evaluate(agent_fn, frame, name):
    correct = 0
    for _, row in frame.iterrows():
        action = agent_fn(row.to_dict())              # the percept is one row, as a dict
        correct += int(action == row["gold"])
    accuracy = correct / len(frame)
    print(f"{name}: {correct}/{len(frame)} correct (accuracy {accuracy:.2f})")
    return accuracy

acc_reflex_easy = evaluate(reflex_agent, easy, "reflex agent, easy subset")
```

</details>


> **Q:** State Russell and Norvig's definition of an agent and explain why a thermostat satisfies it.
<details><summary>Click for answer</summary>

An agent is anything that can be viewed as perceiving its environment through sensors and acting upon that environment through actuators. A thermostat perceives temperature through its sensor and acts on the environment by switching heating on or off, so it implements a (trivial) mapping from percepts to actions. The definition is deliberately inclusive; the interesting distinctions — rationality, autonomy, internal structure — are layered on top of it. Our `reflex_agent` is a thermostat for search results.

</details>


## Part D — Where hand-written rules break

A perfect score on the easy subset — of course: those are the situations the designer anticipated. The `tricky` subset was hand-picked to be adversarial: every row violates one of the assumptions silently baked into the rule table. Run the agent on it and study each failure. Keep four lecture concepts in hand while you read the output:

- **unanticipated situation** — the world leaves the designer's assumptions (no rule fits, or a rule misfires),
- **history matters** — the correct action depends on earlier percepts, which a simple reflex agent cannot remember,
- **multi-agent, adversarial** — other actors deliberately game the rules (SEO spam),
- **performance measure** — relevance and currency are part of report quality, but no keyword rule can express them.


In [ ]:
acc_reflex_tricky = evaluate(reflex_agent, tricky, "reflex agent, tricky subset")

print("\nWhere the rules break:")
for _, row in tricky.iterrows():
    action = ___
    if action != ___:
        print(f"- [{row['domain']}] {row['title'][:70]}")
        print(f"    agent said {action}, gold label is {row['gold']}")


<details>
<summary><b>Click here for the solution</b></summary>

```python
acc_reflex_tricky = evaluate(reflex_agent, tricky, "reflex agent, tricky subset")

print("\nWhere the rules break:")
for _, row in tricky.iterrows():
    action = reflex_agent(row.to_dict())
    if action != row["gold"]:
        print(f"- [{row['domain']}] {row['title'][:70]}")
        print(f"    agent said {action}, gold label is {row['gold']}")
```

</details>


> **📝 Report task R1:** For at least four of the failures listed above, name the underlying reason in lecture vocabulary: (a) an *unanticipated situation* outside the designer's assumptions, (b) *history matters* — the simple reflex agent has no memory, (c) an *adversarial multi-agent* environment gaming the rules, or (d) an aspect of the *performance measure* (relevance, currency) that keyword rules cannot express. Justify each in one sentence. Then state which single failure a **model-based reflex agent** (internal state, same rules otherwise) could fix, and why.
> *No solution is provided — include your answer/code and a short justification in your lab report.*


> **Q:** Why does partial observability force an agent to maintain internal state?
<details><summary>Click for answer</summary>

If a single percept does not reveal the full relevant state of the world, the optimal action can depend on information observed earlier and no longer visible. The agent must therefore accumulate and maintain an internal estimate of the unobserved state — otherwise it can only implement reflexes on the current percept and will act suboptimally whenever history matters (the duplicate above is exactly such a case). In LLM agents this internal state is the growing context buffer.

</details>


## Part E — Replace the rule table with a learned policy

Now delete the hand-written intelligence and put a **single LLM call** in its place: same interface (`percept -> action`), same task — but the condition–action rules are now *implicit in a learned policy*. Nobody wrote them; nobody can enumerate, verify or individually patch them.

Note carefully what this is and is not: still **one call per percept — no tools, no loop, no memory**. Structurally it remains a simple reflex agent; only the reflex is *learned* instead of *authored*.


In [ ]:
TRIAGE_SYSTEM_PROMPT = (
    "You are the source-triage step of a research agent. "
    "The research brief is: 'the current state of solid-state battery manufacturing'. "
    "You see ONE search result. Reply with exactly one word: "
    "KEEP if the source looks trustworthy, relevant to the brief and current, "
    "otherwise DISCARD."
)

def llm_agent(percept):
    '''Same interface as reflex_agent, but the "rule table" is a learned policy.'''
    user_msg = (f"domain: {percept['domain']}\n"
                f"title: {percept['title']}\n"
                f"snippet: {percept['snippet']}")
    resp = ollama.chat(
        model=MODEL,
        messages=[{"role": ___, "content": TRIAGE_SYSTEM_PROMPT},
                  {"role": ___, "content": ___}],
        options={"temperature": ___},        # reproducibility over creativity
    )
    answer = resp["message"]["content"].strip().upper()
    return "KEEP" if ___ else "DISCARD"      # robust parsing of the one-word reply

print(llm_agent(tricky.iloc[0].to_dict()))


> **📝 Report task R2:** Complete the gapped cell above (`llm_agent`). In your report, include your completed code and briefly justify two design choices: why temperature $0.0$ for a classifier, and how your parsing handles replies that are not exactly one word.
> *No solution is provided — include your answer/code and a short justification in your lab report.*


### Rules vs. learned policy, side by side

Run both agents on both subsets and plot the comparison. This calls the model once per row (16–20 calls) — expect a short wait on laptop hardware.


In [ ]:
try:
    acc_llm_easy   = evaluate(___, easy,   "LLM agent, easy subset")
    acc_llm_tricky = evaluate(___, tricky, "LLM agent, tricky subset")

    x = np.arange(2)
    plt.figure(figsize=(6, 3.5))
    plt.bar(x - 0.2, [acc_reflex_easy, acc_reflex_tricky], width=0.4, label="reflex agent")
    plt.bar(x + 0.2, [___, ___], width=0.4, label="LLM agent")
    plt.xticks(x, ["easy subset", "tricky subset"])
    plt.ylabel("accuracy")
    plt.ylim(0, 1.05)
    plt.legend()
    plt.title("Hand-written rules vs. learned policy")
    plt.show()
except Exception as e:
    print("Ollama call failed:", e)
    print("Start Ollama with `ollama serve` and pull the model with `ollama pull qwen2.5:7b`.")


<details>
<summary><b>Click here for the solution</b></summary>

```python
try:
    acc_llm_easy   = evaluate(llm_agent, easy,   "LLM agent, easy subset")
    acc_llm_tricky = evaluate(llm_agent, tricky, "LLM agent, tricky subset")

    x = np.arange(2)
    plt.figure(figsize=(6, 3.5))
    plt.bar(x - 0.2, [acc_reflex_easy, acc_reflex_tricky], width=0.4, label="reflex agent")
    plt.bar(x + 0.2, [acc_llm_easy, acc_llm_tricky], width=0.4, label="LLM agent")
    plt.xticks(x, ["easy subset", "tricky subset"])
    plt.ylabel("accuracy")
    plt.ylim(0, 1.05)
    plt.legend()
    plt.title("Hand-written rules vs. learned policy")
    plt.show()
except Exception as e:
    print("Ollama call failed:", e)
    print("Start Ollama with `ollama serve` and pull the model with `ollama pull qwen2.5:7b`.")
```

</details>


**What you should observe** (results vary between models, quantisations and runs):

- On the **easy** subset the LLM is usually on par with the rules — occasionally slightly worse, because small local models sometimes over- or under-triage clear cases. Hand-written rules are excellent *inside* their assumptions.
- On the **tricky** subset the LLM typically fixes exactly the *semantic* failures: the negation, the adversarial keyword-stuffing, the misfiring "subscribe" marker, the obviously irrelevant `.gov` page. It improvises where the rule table was helpless — the learned policy has no fixed anticipation ceiling.
- The **duplicate still gets through**: `llm_agent` sees one percept at a time and remembers nothing between calls — the LLM is stateless. No amount of prompt cleverness fixes a missing memory. That is the gap that state and the agent loop (Session 02) start to close.


> **Q:** Viewed through Russell and Norvig's five agent program types, why is an LLM agent a strange hybrid?
<details><summary>Click for answer</summary>

Behaviourally it resembles a goal-based or utility-based agent: it pursues explicit goals, predicts consequences and trades off options. Structurally, however, none of the required machinery was authored: there are no hand-written condition-action rules, no explicit transition model and no utility function — all of it is implicit in a learned policy. It thus has the competence profile of the top of the hierarchy without the inspectable structure that defined that hierarchy.

</details>


## Part F — The research agent, in classical vocabulary

From Session 02 onwards we start *building* the course's running example — the **research agent** (brief → search → evaluate sources → draft Markdown report → revise, iterating until the report meets the brief or the budget runs out). Today we describe it with the vocabulary just installed — an exercise you should be able to reproduce in the exam for *any* agent system. Part C was a first taste: your triage agents are miniature versions of its EVALUATE box.


> **📝 Report task R3:** Describe the research agent in classical vocabulary, in your own words (Markdown, ~half a page):
> 1. Give its full **PEAS** specification.
> 2. Classify its environment along all **six dimensions**, justifying each in a few words.
> 3. Give a **BDI reading** — beliefs, desires, intentions — and explain why the agent should *commit* to a report outline instead of restructuring after every new source.
> 4. Name one **reactive-layer** reflex that should run beneath deliberation.
>
> *No solution is provided — include your answer/code and a short justification in your lab report.*


> **Q:** Define beliefs, desires and intentions, emphasising the functional difference between desires and intentions.
<details><summary>Click for answer</summary>

Beliefs are the agent's current, possibly incorrect, information about the world, updated from percepts. Desires are states the agent would like to bring about; they may be mutually inconsistent and carry no commitment. Intentions are desires the agent has committed to: they persist across deliberation cycles, constrain which further options the agent considers, and trigger plan selection and execution. The commitment is the functional difference — desires motivate deliberation, intentions structure action.

</details>


## Part G — Tuning & exploration

No gaps in this part — just play. Two knobs from Part B steer a bare model call: **temperature** and the **system prompt**. Watch how much behaviour they move without touching a single line of "agent" code.

- **Temperature sweep**: rerun the first cell several times. At $T=0$ the replies should be (nearly) identical across repetitions; at $T=1.5$ they scatter. Where would you set the dial for a triage classifier — and for brainstorming report outlines?
- **Personas**: the same percept, three system prompts — the triage decision can flip. The policy lives *in the prompt* as much as in the weights.
- If you have `ipywidgets`, the last cell gives you a temperature slider.


In [ ]:
question = "Name a creative, unusual use for a discarded smartphone battery. One sentence."

for temp in (0.0, 0.7, 1.5):
    print(f"--- temperature = {temp} ---")
    for i in range(3):
        try:
            resp = ollama.chat(model=MODEL,
                               messages=[{"role": "user", "content": question}],
                               options={"temperature": temp, "seed": i})
            print("  ", resp["message"]["content"].strip().replace("\n", " ")[:110])
        except Exception as e:
            print("  Ollama call failed:", e)
            break


In [ ]:
percept = tricky.iloc[0].to_dict()          # the adversarial SEO snippet from Part D

personas = {
    "sceptical reviewer": "You are a sceptical peer reviewer judging one search result. Reply with one word: KEEP or DISCARD.",
    "eager marketer":     "You love exciting products and hate missing out on anything. Reply with one word: KEEP or DISCARD.",
    "default triage":     TRIAGE_SYSTEM_PROMPT,
}

user_msg = (f"domain: {percept['domain']}\n"
            f"title: {percept['title']}\n"
            f"snippet: {percept['snippet']}")

for name, sys_prompt in personas.items():
    try:
        resp = ollama.chat(model=MODEL,
                           messages=[{"role": "system", "content": sys_prompt},
                                     {"role": "user", "content": user_msg}],
                           options={"temperature": 0.0})
        print(f"{name:20s} -> {resp['message']['content'].strip()[:60]}")
    except Exception as e:
        print("Ollama call failed:", e)
        break


In [ ]:
try:
    import ipywidgets as widgets
    from IPython.display import display

    def ask(temperature=0.7):
        resp = ollama.chat(model=MODEL,
                           messages=[{"role": "user",
                                      "content": "In one sentence: is a thermostat an agent?"}],
                           options={"temperature": temperature})
        print(f"T={temperature}:", resp["message"]["content"].strip())

    display(widgets.interactive(ask,
                                temperature=widgets.FloatSlider(min=0.0, max=2.0,
                                                                step=0.1, value=0.7)))
except ImportError:
    print("ipywidgets is not installed - rerun the cells above with different values instead.")


> **Q:** Name three classical ideas that survive in modern LLM agents and the modern guise of each.
<details><summary>Click for answer</summary>

The sense-think-act loop survives as the agent loop: model call, tool execution, observation appended to context. Intention persistence survives as explicit plan management in the scaffold — keeping the current plan in context and discouraging constant replanning. Behavioural layering survives as guardrails: cheap, fast, hand-coded checks running beneath expensive model deliberation. (Also acceptable: state–policy separation as context buffer versus model; the environment taxonomy as a design tool.)

</details>


## Wrap-up

**Takeaways**

- An agent maps percept sequences to actions in an environment. A *simple reflex agent* does so with condition–action rules on the current percept only — and today you watched it fail in all the ways the lecture predicted: unanticipated situations, adversarial percepts, performance-measure aspects rules cannot express, and cases where history matters.
- Replacing the hand-written rule table with a single LLM call swaps an *authored* policy for a *learned* one: it improvises beyond the designer's assumptions (negation, adversarial phrasing) — but nobody can enumerate or patch its rules, and it can still be confidently wrong.
- Architecturally, `llm_agent` is *still* a simple reflex agent: one percept in, one action out, no memory — the duplicate slips through either way. The LLM is stateless between calls; fixing this requires state and a loop.
- Autonomy is a **dial**. Today you sat at its far-left notch: bare completions, no tools, no loop. Every later capability of this course is an explicit, visible step to the right.

**Next week** — Session 02 opens the machine: the **agent loop** itself — the LLM call, tool calls, observations, the context buffer as working state, and the stop conditions that keep the loop from running forever.

**For your lab report**

- [ ] **R1** (Part D): classify at least four reflex-agent failures in lecture vocabulary; name the one failure a model-based reflex agent could fix, and why.
- [ ] **R2** (Part E): your completed `llm_agent` cell, plus a short justification of the temperature and parsing choices.
- [ ] **R3** (Part F): the research agent described in classical vocabulary — PEAS, six environment dimensions, BDI reading, one reactive-layer reflex.
